# Loss Curves by Size (3 Raw Curves Per Size)

Compare the three raw curves within each model size. Each subplot clips to that size's common maximum step so that no plotted curve has missing values in the visible range.


In [1]:
from pathlib import Path
import csv
import math
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'analysis' / 'visualization' / 'loss_curves').is_dir() and (candidate / 'results' / 'figures').is_dir():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PAPER_ROOT = find_project_root()
INPUT_DIR = PAPER_ROOT / 'analysis' / 'visualization' / 'loss_curves' / 'data'
OUTPUT_DIR = INPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_LOG_Y = False
PLOT_DELTA = True
IGNORE_SMOOTH_COLUMNS = True
FIXED_MAX_STEP = 4242
FIXED_MIN_STEP = 600
LINESTYLES = ['-', '--', ':']

csv_paths = sorted(INPUT_DIR.glob('*.csv'))
csv_paths


[PosixPath('/Users/heshuai/Documents/99_Unsorted_Review/2026-05-15_imported_archives/02_bundle_zip/projects/Transformer-Geometry/Transformer-Geometry/Transformer-Geometry/analysis/visualization/loss_curves/data/194m.csv'),
 PosixPath('/Users/heshuai/Documents/99_Unsorted_Review/2026-05-15_imported_archives/02_bundle_zip/projects/Transformer-Geometry/Transformer-Geometry/Transformer-Geometry/analysis/visualization/loss_curves/data/1p4_baseline.csv'),
 PosixPath('/Users/heshuai/Documents/99_Unsorted_Review/2026-05-15_imported_archives/02_bundle_zip/projects/Transformer-Geometry/Transformer-Geometry/Transformer-Geometry/analysis/visualization/loss_curves/data/1p4_gate.csv'),
 PosixPath('/Users/heshuai/Documents/99_Unsorted_Review/2026-05-15_imported_archives/02_bundle_zip/projects/Transformer-Geometry/Transformer-Geometry/Transformer-Geometry/analysis/visualization/loss_curves/data/1p4_xsa.csv'),
 PosixPath('/Users/heshuai/Documents/99_Unsorted_Review/2026-05-15_imported_archives/02_bundl

In [2]:
def clean(text: str) -> str:
    return (text or '').replace('\ufeff', '').strip()

def short_run_name(col: str) -> str:
    parts = [p for p in clean(col).split('/') if p]
    if len(parts) >= 2:
        return parts[-2]
    return clean(col)

def load_raw_series(path: Path):
    with path.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.reader(f))
    if not rows or len(rows[0]) < 2:
        raise ValueError(f'Expected at least 2 columns in {path}')

    size_label = path.stem
    x_name = clean(rows[0][0]) or 'step'
    all_cols = [clean(c) for c in rows[0][1:]]
    series = []
    for col_idx, col in enumerate(all_cols):
        if IGNORE_SMOOTH_COLUMNS and '(smooth)' in col:
            continue
        xs = []
        ys = []
        for row in rows[1:]:
            if len(row) < len(all_cols) + 1:
                continue
            try:
                x = float(clean(row[0]))
                y = float(clean(row[col_idx + 1]))
            except ValueError:
                continue
            xs.append(x)
            ys.append(y)
        if xs:
            series.append({
                'size': size_label,
                'run_name': short_run_name(col),
                'x_name': x_name,
                'x': xs,
                'y': ys,
            })
    return size_label, x_name, series

loaded = []
for path in csv_paths:
    loaded.append(load_raw_series(path))

[(size, x_name, len(series), min(max(s['x']) for s in series)) for size, x_name, series in loaded]

ValueError: min() arg is an empty sequence

In [ ]:
len(series[0]['y'])

In [ ]:
DISPLAY_NAMES_BY_RANK = ['Baseline', 'Residual Rotation', 'Value-Space Rotation']

def clip_series_within_size(series_list, min_step=FIXED_MIN_STEP, max_step=FIXED_MAX_STEP):
    common_max_step = float(max_step)
    common_min_step = float(min_step)
    clipped = []
    for s in series_list:
        pairs = [(x, y) for x, y in zip(s['x'], s['y']) if common_min_step <= x <= common_max_step]
        if not pairs:
            continue
        clipped.append({
            **s,
            'x': [x for x, _ in pairs],
            'y': [y for _, y in pairs],
        })
    clipped.sort(key=lambda s: s['y'][-1], reverse=True)
    renamed = []
    for idx, s in enumerate(clipped):
        name = DISPLAY_NAMES_BY_RANK[idx] if idx < len(DISPLAY_NAMES_BY_RANK) else f'Curve {idx+1}'
        renamed.append({**s, 'display_name': name})
    return common_max_step, renamed

prepared = []
for size, x_name, series in loaded:
    common_max_step, clipped = clip_series_within_size(series)
    prepared.append((size, x_name, common_max_step, clipped))

[(size, common_max_step, len(clipped)) for size, _, common_max_step, clipped in prepared]

In [ ]:
prepared[-2][-1][0]

In [ ]:
def plot_grid(prepared, delta=False, use_log_y=False, figsize=(11.0, 8.2)):
    n = len(prepared)
    cols = 2
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize, squeeze=False)
    axes_flat = axes.flatten()

    for ax in axes_flat[n:]:
        ax.axis('off')

    for ax, (size, x_name, common_max_step, series_list) in zip(axes_flat, prepared):
        for idx, s in enumerate(series_list):
            ys = s['y']
            ylabel = 'val loss'
            if delta:
                base = ys[0]
                ys = [y - base for y in ys]
                ylabel = 'val loss - first point'
            ax.plot(s['x'], ys, linewidth=2.0, linestyle=LINESTYLES[idx % len(LINESTYLES)], label=s.get('display_name', f'{size}-{idx+1}'))
        ax.set_title(f'{size} ({int(FIXED_MIN_STEP)} to {int(common_max_step)})')
        ax.set_xlabel(x_name)
        ax.set_ylabel(ylabel)
        if use_log_y and not delta:
            ax.set_yscale('log')
        if delta:
            ax.axhline(0.0, color='gray', linestyle='--', linewidth=1)
        ax.grid(alpha=0.25, linestyle=':')
        ax.legend(frameon=True, fontsize=8, facecolor='white', edgecolor='#cfcfcf')

    title = 'Loss Curves by Size (3 raw curves each)'
    if delta:
        title += ' (relative to first point)'
    fig.suptitle(title, y=0.995)
    fig.tight_layout()
    return fig


In [ ]:
fig = plot_grid(prepared, delta=False, use_log_y=USE_LOG_Y)
plt.show()

In [ ]:
if PLOT_DELTA:
    fig = plot_grid(prepared, delta=True, use_log_y=USE_LOG_Y)
    plt.show()

In [ ]:
suffix = '_logy' if USE_LOG_Y else ''
abs_path = OUTPUT_DIR / f'loss_csv_by_size_raw3_from_notebook{suffix}.png'
fig = plot_grid(prepared, delta=False, use_log_y=USE_LOG_Y)
fig.savefig(abs_path, dpi=220)
plt.close(fig)
print(abs_path)

if PLOT_DELTA:
    delta_path = OUTPUT_DIR / f'loss_csv_by_size_raw3_delta_from_notebook{suffix}.png'
    fig = plot_grid(prepared, delta=True, use_log_y=USE_LOG_Y)
    fig.savefig(delta_path, dpi=220)
    plt.close(fig)
    print(delta_path)

In [ ]:
import pandas as pd

rows = []
for size, x_name, common_max_step, series_list in prepared:
    for s in series_list:
        for x, y in zip(s['x'], s['y']):
            rows.append({
                'model_size': size,
                'curve_name': s.get('display_name', s.get('run_name', 'unknown')),
                'original_run_name': s.get('run_name', ''),
                'x_name': x_name,
                'step': x,
                'val_loss': y,
                'clip_min_step': FIXED_MIN_STEP,
                'clip_max_step': common_max_step,
            })

clean_df = pd.DataFrame(rows)
excel_path = OUTPUT_DIR / 'loss_csv_by_size_raw3_cleaned.xlsx'
csv_path = OUTPUT_DIR / 'loss_csv_by_size_raw3_cleaned.csv'
clean_df.to_csv(csv_path, index=False)
print(csv_path)
try:
    clean_df.to_excel(excel_path, index=False)
    print(excel_path)
except ModuleNotFoundError:
    print('openpyxl not installed, skip xlsx export')
clean_df.head()


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd


# =========================
# File-first configuration
# =========================
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'analysis' / 'visualization' / 'loss_curves').is_dir() and (candidate / 'results' / 'figures').is_dir():
            return candidate
    raise RuntimeError(f'Could not locate project root from {start}')


PAPER_ROOT = find_project_root()
INPUT_DIR = PAPER_ROOT / 'analysis' / 'visualization' / 'loss_curves' / 'data'
CLEANED_CSV = INPUT_DIR / 'loss_csv_by_size_raw3_cleaned.csv'
RAW_LOSS_DIR = INPUT_DIR
OUTPUT_DIR = RAW_LOSS_DIR / 'plots'
FIG_DPI = 220
USE_LOG_Y = False
PLOT_DELTA = True
FIGSIZE = (11.0, 8.0)
LINESTYLES = {
    "Baseline": "-",
    "Residual Rotation": "--",
    "Value-Space Rotation": ":",
}
SIZE_ORDER = ["194m", "296m", "436m", "528m"]
SIZE_COLORS = {
    "194m": "#1f77b4",
    "296m": "#ff7f0e",
    "436m": "#2ca02c",
    "528m": "#d62728",
}
FIXED_MIN_STEP = 600
FIXED_MAX_STEP = 4242


def _plot_from_cleaned_csv(df: pd.DataFrame, out_prefix: str) -> list[Path]:
    outputs: list[Path] = []
    fig, axes = plt.subplots(2, 2, figsize=FIGSIZE, squeeze=False)
    axes_flat = axes.flatten()

    for ax, size in zip(axes_flat, SIZE_ORDER):
        sdf = df[df["model_size"] == size].copy()
        if sdf.empty:
            ax.axis("off")
            continue
        for curve_name in ["Baseline", "Residual Rotation", "Value-Space Rotation"]:
            cdf = sdf[sdf["curve_name"] == curve_name].sort_values("step")
            if cdf.empty:
                continue
            ax.plot(cdf["step"], cdf["loss"], label=curve_name, linewidth=2.0)
        ax.set_title(size)
        ax.grid(alpha=0.25)
        ax.legend()

    fig.tight_layout()
    out_path = OUTPUT_DIR / f"{out_prefix}.png"
    fig.savefig(out_path, dpi=FIG_DPI, bbox_inches="tight")
    outputs.append(out_path)
    plt.close(fig)
    return outputs
